# Unsupervised Visual Anomaly Discovery for Industrial Quality Inspection
## Comprehensive Manifold & Clustering Walkthrough: 8-Line Industrial Suite
**Author:** Gayatri U & Tanya Tejaswi (Unsupervised Learning & Dimensionality Reduction)

This notebook demonstrates the end-to-end unsupervised pipeline across all **8 Industrial Manufacturing Lines**:
1. **pcb1** (Mainboard Traces - Amazon VisA)
2. **pcb2** (Micro-Soldering - Amazon VisA)
3. **pcb3** (IC Component Array - Amazon VisA)
4. **pcb4** (High-Density Board - Amazon VisA)
5. **capsules** (Medical Capsules - Amazon VisA)
6. **bottle** (Bottling & Cap Line - MVTec AD)
7. **cable** (Wiring Harness - MVTec AD)
8. **steel** (Hot-Rolled Strips - NEU Surface Defect)

---
### Pipeline Architecture:
1. **Deep Feature Ingestion & Standardization** (2,048-dimensional ResNet-50 manifold)
2. **Dimensionality Reduction via Principal Component Analysis (PCA)**
3. **K-Means Inertia Elbow Optimization ($K=8$ Manufacturing Clusters)**
4. **Density-Based Defect Isolation (DBSCAN & HDBSCAN)**
5. **2D Manifold Visualizations (t-SNE Projections with Defect Callouts)**
6. **Quantitative Intrinsic & Extrinsic Cluster Validation**

In [ ]:
import os
import sys
import tempfile
os.environ["MPLCONFIGDIR"] = tempfile.gettempdir()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Ensure project root is accessible
sys.path.append("..")

from src.dimensionality_reduction import scale_features, apply_pca, compute_explained_variance, find_optimal_components
from src.clustering import run_kmeans, compute_elbow_inertias, run_dbscan, run_hdbscan, extract_anomaly_indices, get_cluster_summary
from src.visualization import compute_tsne_embedding, plot_tsne_clusters, plot_pca_scree, plot_elbow_curve
from src.evaluation import compute_clustering_metrics

print("✅ All modules loaded successfully!")

### 1. Ingestion of 2,048-Dimensional Deep Feature Embeddings across 8 Lines
We load pre-extracted deep features and metadata indexing 976 items across the 8 industrial categories.

In [ ]:
# Load embeddings and metadata
emb_path = Path("../data/embeddings/image_embeddings.npy")
meta_path = Path("../data/embeddings/metadata.csv")

if not emb_path.exists():
    emb_path = Path("../datasets/features.npy")

features = np.load(emb_path)
meta_df = pd.read_csv(meta_path) if meta_path.exists() else None

print(f"📊 Feature Matrix Shape: {features.shape[0]} samples x {features.shape[1]} dimensions")
if meta_df is not None:
    print(f"🏭 Breakdown across 8 Industrial Lines:")
    print(meta_df['category'].value_counts())

### 2. Standardization & Principal Component Analysis (PCA)
Standardize features to zero-mean and unit variance, then decompose variance via PCA retaining 95% of total variance.

In [ ]:
# Standardize features
scaled_features, scaler = scale_features(features)

# Retain 95% of total variance
reduced_features, pca_model = apply_pca(scaled_features, explained_variance_ratio=0.95)
ind_var, cum_var = compute_explained_variance(pca_model)

print(f"Reduced from {features.shape[1]} dimensions down to {reduced_features.shape[1]} principal components.")
print(f"Cumulative Explained Variance Retained: {cum_var[-1]*100:.2f}%")

# Generate Scree Plot
os.makedirs("../results/plots", exist_ok=True)
fig_scree = plot_pca_scree(ind_var, cum_var, threshold=0.95, save_path="../results/plots/pca_scree.png")
plt.show()

### 3. K-Means Clustering & Inertia Elbow Curve Optimization ($K=2 \dots 12$)
We search for the mathematical inflection point in within-cluster sum-of-squares (WCSS) to discover the natural cluster structure matching our 8 manufacturing categories.

In [ ]:
# Compute inertia across cluster counts 2 through 12
k_range = list(range(2, 13))
k_vals, inertias = compute_elbow_inertias(reduced_features, k_range=k_range)

fig_elbow = plot_elbow_curve(k_vals, inertias, save_path="../results/plots/kmeans_elbow.png")
plt.show()

# Fit K-Means with optimal K=8
kmeans_labels, km_model = run_kmeans(reduced_features, n_clusters=8, random_state=42)
km_metrics = compute_clustering_metrics(reduced_features, kmeans_labels)

print("🏆 K-Means Clustering Metrics (K=8):")
for metric_name, val in km_metrics.items():
    print(f"  - {metric_name}: {val}")

### 4. Density-Based Clustering & Unsupervised Defect Discovery (DBSCAN & HDBSCAN)
Density-based clustering partitions the high-density normal component clusters and flags peripheral defective outliers as anomaly noise ($label = -1$).

In [ ]:
# Run DBSCAN
dbscan_labels, _ = run_dbscan(reduced_features, eps=25.0, min_samples=5)
db_summary = get_cluster_summary(dbscan_labels)
db_metrics = compute_clustering_metrics(reduced_features, dbscan_labels, exclude_noise=True)

print("🔍 DBSCAN Discovery Summary:")
print(f"  - Discovered Cohesive Clusters: {db_summary['num_clusters']}")
print(f"  - Isolated Defective Anomalies: {db_summary['num_anomalies']} ({db_summary['anomaly_percentage']:.2f}%)")
print(f"  - Silhouette Score:             {db_metrics.get('silhouette_score', 0):.4f}")

# Run HDBSCAN (if available)
try:
    hdbscan_labels, probs, _ = run_hdbscan(reduced_features, min_cluster_size=5)
    hdb_summary = get_cluster_summary(hdbscan_labels)
    print(f"\n🔍 HDBSCAN: Found {hdb_summary['num_clusters']} clusters and {hdb_summary['num_anomalies']} anomaly outliers.")
except Exception as e:
    print(f"HDBSCAN note: {e}")

### 5. 2D Manifold Visualizations (t-SNE Projections)
We compute a non-linear 2D t-SNE projection to visualize the separation of the 8 industrial categories and the isolation of defective items.

In [ ]:
# Compute 2D t-SNE embedding
tsne_2d = compute_tsne_embedding(reduced_features, n_components=2, perplexity=30.0, random_state=42)

# Plot K-Means Discovered Clusters
fig_km = plot_tsne_clusters(
    tsne_2d,
    kmeans_labels,
    title="t-SNE: Discovered K-Means Manufacturing Clusters (K=8)",
    save_path="../results/plots/tsne_kmeans.png"
)
plt.show()

# Plot DBSCAN Anomaly Discovery
fig_db = plot_tsne_clusters(
    tsne_2d,
    dbscan_labels,
    title="t-SNE: DBSCAN Density Anomaly Discovery (Noise = Anomalies)",
    save_path="../results/plots/tsne_dbscan.png"
)
plt.show()

### 6. Benchmark Summary & Reporting
Save the intrinsic validation metrics across all clustering approaches to `reports/` and `results/metrics/`.

In [ ]:
# Compile benchmark table
benchmark_records = [
    {
        "algorithm": "K-Means (K=8)",
        "clusters": km_metrics["evaluated_clusters"],
        "silhouette": round(km_metrics["silhouette_score"], 4),
        "davies_bouldin": round(km_metrics["davies_bouldin_index"], 4),
        "calinski_harabasz": round(km_metrics["calinski_harabasz_index"], 2)
    },
    {
        "algorithm": "DBSCAN (eps=25.0)",
        "clusters": db_metrics["evaluated_clusters"],
        "silhouette": round(db_metrics["silhouette_score"], 4),
        "davies_bouldin": round(db_metrics["davies_bouldin_index"], 4),
        "calinski_harabasz": round(db_metrics["calinski_harabasz_index"], 2)
    }
]

benchmark_df = pd.DataFrame(benchmark_records)
os.makedirs("../reports", exist_ok=True)
benchmark_df.to_csv("../reports/clustering_benchmark.csv", index=False)
print("🏆 Final Clustering Benchmark Scoreboard:")
print(benchmark_df.to_string(index=False))